# Global AlphaEarth Building Height Estimation

This notebook applies all seven regional MLP models in Earth Engine and mosaics them by model-region masks. Use it for global or cross-region inference without manually choosing `region_key`.

## 1. Imports and Earth Engine initialization

In [ ]:
import ee

try:
    import geemap
except ImportError:
    geemap = None

import model as ae_height_model

try:
    ee.Initialize(project='YOUR GEE PROJECT ID')
except Exception:
    ee.Authenticate()
    ee.Initialize()

ae_height_model.available_regions()

## 2. Load AlphaEarth annual embedding

In [ ]:
year = 2023

alpha_earth = (
    ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL')
    .filterDate(f'{year}-01-01', f'{year + 1}-01-01')
    .mosaic()
    .select(list(ae_height_model.DEFAULT_BAND_NAMES))
)

alpha_earth.bandNames().getInfo()[:5]

## 3. Build regional-model mosaic for an ROI

`predict_global_height()` runs every regional model and masks each prediction before mosaicking. For interactive testing, pass `clip_geometry=roi` so Earth Engine only builds the graph over your area of interest. The default `mask_source="lonlat"` uses coarse non-empty lon/lat masks and avoids empty-boundary errors.


In [ ]:
# South China example bbox: [xmin, ymin, xmax, ymax]
bbox = [109.863281, 20.468189, 118.223877, 25.502785]
roi = ae_height_model.bbox_to_geometry(bbox)

height_roi = ae_height_model.predict_global_height(
    alpha_earth_image=alpha_earth,
    output_name="building_height_m",
    clip_geometry=roi,
    mask_source="lonlat",
).clip(roi)

region_candidates = ae_height_model.region_candidates_from_bbox(bbox)
print("Region candidates:", region_candidates)
height_roi


## 4. Preview


In [ ]:
vis = {
    'min': 0,
    'max': 20,
    'palette': ['0b1d3a', '2066ac', '59c1bd', 'f2e85e', 'f28e2b', 'b11226'],
}

if geemap is not None:
    m = geemap.Map()
    m.centerObject(roi, 7)
    m.addLayer(height_roi, vis, f'Global regional-model height {year}')
    m.addLayer(roi, {}, 'ROI')
    m
else:
    print('Install geemap for interactive display, or continue to export the image.')

In [ ]:
m

## 5. Export ROI or global image

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=height_roi.float(),
    description=f'ae_global_regional_height_roi_{year}',
    folder='earthengine_exports',
    fileNamePrefix=f'ae_global_regional_height_roi_{year}',
    region=roi,
    scale=10,
    maxPixels=1e13,
)

# Uncomment to start the ROI export.
# task.start()
# task.status()

In [ ]:
# Global export template. This is much heavier than ROI export.
# global_height = ae_height_model.predict_global_height(
#     alpha_earth_image=alpha_earth,
#     output_name="building_height_m",
# )
# global_task = ee.batch.Export.image.toAsset(
#     image=global_height.float(),
#     description=f"ae_global_regional_height_{year}",
#     assetId=f"users/your_name/ae_global_regional_height_{year}",
#     region=ee.Geometry.Rectangle([-180, -60, 180, 85], proj="EPSG:4326", geodesic=False),
#     scale=10,
#     maxPixels=1e13,
# )
# global_task.start()
# global_task.status()
